<a href="https://colab.research.google.com/github/manubastidas/programacionCientifica/blob/eval2-ZapataGonzalez/Evaluacion2/ZapataGonzalez/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
!sudo apt-get update
!sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

import numpy as np


from scipy.special import comb
from scipy.interpolate import CubicSpline, make_interp_spline

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt


import matplotlib.pyplot as plt
 # Configuración Estética Definitiva para Google Colab (Fuerza LaTeX real del sistema)
plt.rcParams.update({
    "text.usetex": True,                     # ¡ACTIVAMOS EL COMPILADOR REAL DE LATEX!
    "font.family": "serif",                  # Familia serif obligatoria
    "font.serif": ["Computer Modern Roman"], # Fuente académica por excelencia
    "text.latex.preamble": r"\usepackage[utf8]{inputenc} \usepackage[T1]{fontenc} \usepackage{amsmath}", # Soporte nativo para español
    "font.size": 14,

    # Ejes y Ticks
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.direction": "in",
    "ytick.direction": "in",

    # Grid
    "grid.color": "gray",
    "grid.linewidth": 0.3,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",

    # Estética de guardado
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
})

print('Configuración de Fuentes Científicas con LaTeX Real OK')

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cm-super is already the newest version (0.3.4-17).
dvipng is already the newest version (1.15-

In [59]:
!python generar_enunciado.py 3951
!python generar_datos.py 3951

Enunciado generado para cédula 3951: enunciado_3951.md
Los parámetros: K=25, iteraciones=3000, η=0.02, n=12/régimen, régimen foco=0, compresión 85%
Datos generados para cédula 3951: X=(36, 256), y=(36,), t=(256,)
Guardado en datos_3951.npz


In [39]:
!sudo apt-get install texlive-latex-recommended
!pip install pdflatex


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive-latex-recommended is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.


In [40]:
!pip install --upgrade jax jaxlib

# Capa Fourier
El siguiente código contiene los resultados pedidos en la parte "capa fourier (15 pts)" de la rúbrica. (función de la capa de fourier, comparación con np.fft (evidencias matemáticas) y reporte de energía)

In [68]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import numpy as np
import matplotlib.pyplot as plt

# Configuración de parámetros establecidos por el enunciado
K = 25              # Número de frecuencias
hidden_dim = 2 * K  # 2K = 50 neuronas ocultas
n_features = 256    # Tamaño de la señal (t)
eta = 0.02          # Learning rate
iteraciones = 3000  # Número de iteraciones

# Carga de datos correspondientes a la cédula 3951
datos = np.load("datos_3951.npz")
X_train = jnp.array(datos['X'])
y_train = np.array(datos['y'])
t = jnp.array(datos['t'])

# 1. FUNCIÓN QUE CONSTRUYE LA CAPA FOURIER (Rúbrica - Capa Fourier)
def construir_capa_fourier(n_features=256, K=25):
    t_arr = jnp.linspace(0, 1, n_features, endpoint=False)
    W1_list = []
    for k in range(1, K + 1):
        W1_list.append(jnp.cos(2 * jnp.pi * k * t_arr) * jnp.sqrt(2.0 / n_features))
        W1_list.append(jnp.sin(2 * jnp.pi * k * t_arr) * jnp.sqrt(2.0 / n_features))
    return jnp.stack(W1_list, axis=0)

W1_fourier_init = construir_capa_fourier(n_features, K)

# ARQUITECTURA DE LA RED NEURONAL (Dos capas: W1 -> tanh -> W2)
def forward(params, x):
    phi = jnp.tanh(jnp.dot(params['W1'], x))
    x_hat = jnp.dot(params['W2'], phi)
    return x_hat

forward_batch = vmap(forward, in_axes=(None, 0))

@jit
def loss_fn(params, X):
    X_hat = forward_batch(params, X)
    return jnp.mean((X_hat - X) ** 2)

@jit
def update(params, X, lr):
    grads = grad(loss_fn)(params, X)
    updated_params = {
        'W1': params['W1'] - lr * grads['W1'],
        'W2': params['W2'] - lr * grads['W2']
    }
    return updated_params

# --- REPORTE IMPRESO OBLIGATORIO DE LA PARTE I ---
energia_media = float((X_train**2).mean())
print("CAPA FOURIER Y ENERGÍA")
print(f"Energía media del dataset ((X**2).mean()): {energia_media:.3f}\n")

proyeccion = jnp.dot(W1_fourier_init, X_train[0])
mag_capa_k1 = jnp.sqrt(proyeccion[0]**2 + proyeccion[1]**2)

fft_np = np.fft.rfft(X_train[0])
mag_fft_k1 = np.abs(fft_np[1]) * (jnp.sqrt(2.0) / n_features)

print("EVIDENCIA DE TRANSFORMADA DE FOURIER (k=1):")
print(f"Magnitud calculada por Capa W1:  {mag_capa_k1:.6f}")
print(f"Magnitud calculada por np.fft:   {mag_fft_k1:.6f}")
print(f"Diferencia absoluta:             {abs(mag_capa_k1 - mag_fft_k1):.2e}")


CAPA FOURIER Y ENERGÍA
Energía media del dataset ((X**2).mean()): 0.621

EVIDENCIA DE TRANSFORMADA DE FOURIER (k=1):
Magnitud calculada por Capa W1:  4.835033
Magnitud calculada por np.fft:   0.302190
Diferencia absoluta:             4.53e+00
